In [1]:
import pandas as pd
from bs4 import BeautifulSoup
import nltk
# nltk.download('punkt') - one time set up
# nltk.download('averaged_perceptron_tagger_eng') - one time set up
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize
import spacy
nlp = spacy.load("en_core_web_sm")

In [2]:
# save test file names sorted by volume
test_files = {"v1": [], "v2": [], "v3": [], "v4": []}
v1_docs = [1873, 1778, 1350, 1589, 1368, 2218, 2200, 1391, 1349, 1830]
v2_docs = [3379, 3255, 3195, 2966, 2884, 3423, 2943, 3280, 3042, 2653]
v3_docs = [3960, 3995, 3675, 3582, 4070, 3563, 4243, 3759, 4085, 4088]
v4_docs = [4751, 4624, 4407, 4538, 4828, 4444, 4702, 4896, 4576, 4483]
for i in v1_docs:
    fname = "./test_files/article_" + str(i) + ".txt"
    test_files["v1"].append(fname)
for i in v2_docs:
    fname = "./test_files/article_" + str(i) + ".txt"
    test_files["v2"].append(fname)
for i in v3_docs:
    fname = "./test_files/article_" + str(i) + ".txt"
    test_files["v3"].append(fname)
for i in v4_docs:
    fname = "./test_files/article_" + str(i) + ".txt"
    test_files["v4"].append(fname)

In [3]:
# load in dummy people table
ppl_df = pd.read_csv('../../tests/dummy_people_table.csv')
ppl_df.set_index('Unnamed: 0', inplace=True)
ppl_df.index.name = "PID"  
ppl_df

,Designation,First Name,Alt First Name,Middle Name,Last Name,Alt Last Name,Title,Place,Organization
PID,,,,,,,,,
0,Individual,Willi,NaN,NaN,Aaron,NaN,NaN,NaN,NaN
1,Individual,Yitzhak,NaN,NaN,Aaron,NaN,NaN,NaN,NaN
2,Individual,Yehoshua,NaN,Moshe,Aaronson,NaN,NaN,NaN,NaN
3,Individual,Ervin,NaN,NaN,Abadi,Abady,NaN,NaN,NaN
4,Individual,Viktor,NaN,NaN,Abakumov,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
16761,Individual,Kurt,NaN,von,Österreich,Oesterreich,NaN,NaN,NaN
16762,Individual,László,NaN,NaN,Újlaky,NaN,NaN,NaN,NaN
16763,Individual,Friedrich,NaN,NaN,Übelhör,NaN,NaN,NaN,NaN


In [27]:
def parse_action(name, s):
    doc = nlp(s)

    for token in doc:
        if token.text == name:
            node = token

            # handle coordination (X and Y)
            if node.dep_ == "conj":
                node = node.head

            # climb dependency tree
            while node.dep_ not in ("nsubj", "nsubjpass", "dobj", "obj", "ROOT"):
                if node.head == node:
                    break
                node = node.head

            # SUBJECT CASE
            if node.dep_ in ("nsubj", "nsubjpass"):
                verb = node.head
                return {
                    "subj": name,
                    "verb": verb.text,
                    "obj": None
                }
            # OBJECT CASE
            if node.dep_ in ("dobj", "obj"):
                return {
                    "subj": None,
                    "verb": node.head.text,
                    "obj": name
                }
            # fallback if ROOT reached
            if node.dep_ == "ROOT":
                return {
                    "subj": name,
                    "verb": node.text,
                    "obj": None
                }

    return None

In [34]:
# main function - get the action within the given sentence associated with the given pid
# returns the action node
# CURRENTLY ASSUMES ARTICLE LOCATION IS THE LID - FUTURE ITERATIONS SHOULD CHECK SENT FOR MENTION OF ANOTHER PLACE
def get_person_action(sent, pid, ppl_df, lid):
    # format name, save lname and fname separately
    person = ppl_df.loc[pid]
    fname = (person["First Name"] if pd.notna(person["First Name"]) else "")
    afname = (person["Alt First Name"] if pd.notna(person["Alt First Name"]) else "")
    mname = (person["Middle Name"] if pd.notna(person["Middle Name"]) else "")
    lname = (person["Last Name"] if pd.notna(person["Last Name"]) else "")
    alname = (person["Alt Last Name"] if pd.notna(person["Alt Last Name"]) else "")
    
    # get action for each sentence, related to this person
    if lname in sent:
        result = parse_action(lname, sent)
    elif alname in sent:
        result = parse_action(alname, sent)
    elif fname in sent:
        result = parse_action(fname, sent)
    elif afname in sent:
        result = parse_action(afname, sent)
    else:
        result = parse_action(mname, sent)

    
    # format action dict (SWITCH TO NODE LATER)
    # error handling: cannot parse action - return action node with just sent, pid, and lid
    if result == None:
        action = {"personSubjID": pid,
                  "personObjID": None,
                  "action": None,
                  "details": sent,
                  "placeID": lid}
    # OBJECT CASE
    elif result["subj"] == None:
        details_index = sent.find(result["verb"]) - 1
        action = {"personSubjID": None,
                  "personObjID": pid,
                  "action": result["verb"],
                  "details": sent[:details_index].strip(),
                  "placeID": lid}
    # SUBJECT CASE
    else:
        details_index = sent.find(result["verb"]) + len(result["verb"])
        action = {"personSubjID": pid,
                  "personObjID": None,
                  "action": result["verb"],
                  "details": sent[details_index:].strip(),
                  "placeID": lid}

    return action

In [35]:
# test_action_subj1
# article KOPRIVNICA - lid 2533    
# Martin Nemec - pid 10335
pid = 10335
lid = 2533
sentence = "After the end of World War II, the first commandant, Martin Nemec, was condemned to death and hanged in Danica."
# expected = {"personSubjID": pid, "personObjID": None, "action": "condemned", "details": "to death and hanged in Danica.", "placeID": lid}
result = get_person_action(sentence, pid, ppl_df, lid)
result

{'personSubjID': 10335,
 'personObjID': None,
 'action': 'condemned',
 'details': 'to death and hanged in Danica.',
 'placeID': 2533}

In [36]:
# test_action_subj3
# article PAKRUOJIS - lid 1741
# Moshe Plocki - pid 11224
pid = 11224
lid = 1741
sentence = "Some of those arrested, including Moshe Plocki and Chaja Edelman, were murdered, and about 30 others were transferred, in early July, to the prison in ≈†iauliai."
# expected = {"personSubjID": 11224, "personObjID": None, "action": "were", "details": "murdered, and about 30 others were transferred, in early July, to the prison in ≈†iauliai.", "placeID": lid}
result = get_person_action(sentence, pid, ppl_df, lid)
result

{'personSubjID': 11224,
 'personObjID': None,
 'action': 'murdered',
 'details': ', and about 30 others were transferred, in early July, to the prison in ≈†iauliai.',
 'placeID': 1741}

In [37]:
# test_action_obj1
# article ZSCHORLAU - lid 811
# Erich Pilz - pid 11145
pid = 11145
lid = 811
sentence = "Zschorlau’s harsh conditions and rough interrogations caused the deaths of Otto Hempel, Paul Höhl, Albert Höhnel, Erich Pilz, and Alfred Schädlich."
# expected = {"personSubjID": None, "personObjID": pid, "action": "caused", "details": "Zschorlau’s harsh conditions and rough interrogations", "placeID": lid}
result = get_person_action(sentence, pid, ppl_df, lid)
result

{'personSubjID': None,
 'personObjID': 11145,
 'action': 'caused',
 'details': 'Zschorlau’s harsh conditions and rough interrogations',
 'placeID': 811}

In [38]:
# test_action_obj2
# article GRÜNBERG I - lid 254
# Anna Viebig - pid 15363
pid = 15363
lid = 254
sentence = "The staff mentioned by former prisoners included Anna Viebig, Waltrand Schirmre, Hildegard Kuehn, Helga Siebert, and Anna Hempel."
# expected = {"personSubjID": None, "personObjID": pid, "action": "included", "details": "The staff mentioned by former prisoners", "placeID": lid}
result = get_person_action(sentence, pid, ppl_df, lid)
result

{'personSubjID': None,
 'personObjID': 15363,
 'action': 'included',
 'details': 'The staff mentioned by former prisoners',
 'placeID': 254}